# Factoring with modular exponentiation

This worked example uses the period of modular exponentiation to factor 221. The period search here is classical; the reversible circuit below is the modular exponentiation subroutine used in quantum period finding.

Adapted from the Qualtran v0.7.0 tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

[Shor's factoring algorithm and its resource setting](https://arxiv.org/abs/1905.09749)

In [ ]:
from collections import Counter
from math import gcd

from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.modular_exponentiation import ModularExponentiation
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit

N = 13 * 17
base = 8
assert gcd(base, N) == 1
modexp = ModularExponentiation.for_shor(N, base)
assert (modexp.exponent_width, modexp.output_width) == (16, 8)
N, base

## Find the period

Visit the first twenty exponents and look for repeated returns to 1.

In [ ]:
reference = [(e, pow(base, e, N)) for e in range(20)]
period_hits = [e for e, value in reference if value == 1]
assert period_hits == [0, 8, 16]
period = period_hits[1] - period_hits[0]
print(*reference, sep="\n")
period

## Recover the factors

For an even period, the two greatest common divisors below give the factors when the half-period power is neither 1 nor −1 modulo `N`.

In [ ]:
assert period % 2 == 0
midpoint = pow(base, period // 2, N)
assert midpoint not in (1, N - 1)
factors = gcd(midpoint - 1, N), gcd(midpoint + 1, N)
assert set(factors) == {13, 17}
factors

## Inspect the reversible subroutine

The through exponent is big-endian, and the result word starts at zero. The circuit prepares 1, then visits the exponent from its least significant bit, applying controlled multiplications by successive squared constants. The symbolic declaration groups all multiplications, including those with constant 1.

In [ ]:
display(SVG(field_signature_svg((("ModExp(8, 221)", modexp.source_signature, {}),))))
graph = modexp.source_call_graph()
assert graph.leaf_counts == {"CModMulK": 16, "|1>": 1}
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(Markdown(format_call_graph_markdown(graph.root, graph.direct_calls, graph.leaf_counts)))

In [ ]:
values = [modexp.classical_values(e) for e in range(20)]
assert values == reference
multiplier = base % N
multipliers = []
for _ in range(modexp.exponent_width):
    multipliers.append(multiplier)
    multiplier = multiplier * multiplier % N
multipliers

## Append a small circuit

The three-bit example computes powers of 2 modulo 3. Caller-owned workspace is clean after each arithmetic operation. The program is built here for inspection; the [audit](../../docs/development/RSA_MODEXP_FILE_AUDIT.md) records full-state and measurement-branch verification.

In [ ]:
small = ModularExponentiation(2, 3, 3, 2)
q, c = small.workspace_requirements()
exponent, target = (0, 1, 2), (3, 4)
circuit = DynamicCircuit(5 + q).append(small.make_on(
    exponent=exponent, target=target, workspace=tuple(range(5, 5 + q)),
    classical_bits=tuple(range(c)),
))
ir = circuit.to_ir()
observed = Counter(g.name for g in ir.instructions)
assert observed["measure"] == 144 and ir.n_wires == 18
{"bound_wires": ir.n_wires, "measurements": observed["measure"]}